# Multi-Objective Evaluation
* Monte Carlo Hypervolume (HV) estimation from a folder of images
* Radar charts
* Statistics over generations

Notebook Version: 0.10.0 (30.03.2026)
* Rename to `mo_evaluation.ipynb`
* Add radar charts and statistics


In [ ]:
from pathlib import Path
from typing import  List, Tuple, Optional
import numpy as np
from PIL import Image
from evolutionary_imaging.image_base import ImageSolutionData
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# ---------- helpers ----------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def list_images(folder: str, recursive: bool = False) -> List[Path]:
    p = Path(folder)
    it = p.rglob("*") if recursive else p.glob("*")
    return sorted([q for q in it if q.suffix.lower() in IMG_EXTS])

def normalize_to_unit(F: np.ndarray,
                      mins: Optional[np.ndarray] = None,
                      maxs: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if mins is None or maxs is None:
        mins = F.min(axis=0)
        maxs = F.max(axis=0)
    rng = np.maximum(maxs - mins, 1e-12)
    return np.clip((F - mins) / rng, 0.0, 1.0), mins, maxs

def first_front_indices(F_for_minimization: np.ndarray) -> np.ndarray:
    """Return indices of the nondominated front; input must be in minimization sense."""
    nd = NonDominatedSorting().do(F_for_minimization, n_stop_if_ranked=len(F_for_minimization))[0]
    return np.asarray(nd, dtype=int)

# ---------- main: MC HV over all images in a folder ----------
def hv_mc_for_folder(folder: str,
                     evaluator,
                     *,
                     samples: int = 200_000,
                     batch: int = 50_000,
                     seed: int = 42,
                     mins: Optional[np.ndarray] = None,
                     maxs: Optional[np.ndarray] = None,
                     recursive: bool = False,
                     print_rows: bool = True):
    """
    Evaluate all images in `folder` with your evaluator and compute Monte-Carlo HV.
    Returns (hv, (ci_low, ci_high), F, paths, mins, maxs).

    - Normalizes to [0,1] per objective (use `mins/maxs` to keep a fixed scale across runs).
    - Prunes to the nondominated set before HV (for speed; HV is unchanged).
    """

    paths = list_images(folder, recursive=recursive)
    if not paths:
        raise ValueError(f"No images found in: {folder}")

    fitness_rows: List[np.ndarray] = []
    kept_paths: List[Path] = []

    for path in paths:
        try:
            img = Image.open(path).convert("RGB")
            # Your evaluator expects ImageSolutionData(images=[PIL.Image]) as input:
            f = evaluator.evaluate(ImageSolutionData(images=[img]))
            f = np.asarray(f, dtype=float)
            if f.ndim != 1:
                f = f.ravel()
            fitness_rows.append(f)
            kept_paths.append(path)
            if print_rows:
                print(f"{path}: {f.tolist()} | sum={float(np.sum(f))}")
        except Exception as e:
            # Skip unreadable / unevaluable images
            print(f"[skip] {path} -> {e}")

    if not fitness_rows:
        raise ValueError("No images could be evaluated.")

    F = np.vstack(fitness_rows)        # (N, M), maximize
    # Normalize
    F01, mins, maxs = normalize_to_unit(F, mins, maxs)

    # De-duplicate close points (optional but speeds up MC)
    F01 = np.unique(np.round(F01, 6), axis=0)

    # Keep only nondominated points (in maximize -> convert to minimize by negation)
    nd_idx = first_front_indices(-F01)   # NonDominatedSorting expects minimization
    F01_nd = F01[nd_idx, :]

    # Monte-Carlo HV on [0,1]^M (maximize)
    rng = np.random.default_rng(seed)
    M = F01_nd.shape[1]
    taken = 0
    total = 0
    while total < samples:
        b = min(batch, samples - total)
        U = rng.random((b, M))                                        # Uniform in [0,1]^M
        dom = (F01_nd[None, :, :] >= U[:, None, :]).all(axis=2).any(axis=1)
        taken += int(dom.sum())
        total += b
    hv = taken / total
    se = max(1e-12, np.sqrt(hv * (1 - hv) / samples))
    ci = (hv - 1.96 * se, hv + 1.96 * se)

    # Print top 3 by sum
    if print_rows:
        sums = F.sum(axis=1)
        top3_idx = np.argsort(sums)[-3:][::-1]
        for idx in top3_idx:
            print(f"[top-sum] {kept_paths[idx]}: {F[idx].tolist()} | sum={float(sums[idx])}")

    return hv, ci, F, kept_paths, mins, maxs

In [ ]:
from evolutionary_imaging.evaluators import MultiCLIPIQAEvaluator

metrics = ("quality", "sharpness", "contrast", "colorfullness", "brightness",
 "natural", "real", "beautiful", "new", "complexity")
n = len(metrics)
evaluator = MultiCLIPIQAEvaluator(metrics=metrics, clip_model='clip_iqa')

## Hypervolume

In [ ]:
mins = np.zeros(n, dtype=float)
maxs = np.ones(n, dtype=float)

hv, ci, F, paths, mins, maxs = hv_mc_for_folder(
    folder="119",
    evaluator=evaluator,
    samples=800_000,
    print_rows=True,
    mins=mins,
    maxs=maxs,
)
print(f"HV ≈ {hv:.5f}  (95% CI: {ci[0]:.5f}, {ci[1]:.5f})")

## Statistics

In [ ]:
from evolutionary.statistics import StatisticsTracker
from PIL import Image
from evolutionary.evolution_base import SolutionCandidate
from evolutionary_imaging.image_base import ImageSolutionData
import glob
import gc
import pickle
import os
import evolutionary_imaging.processing as ip
from tqdm import tqdm

mo_stats = StatisticsTracker()
stats_file = "results/mo_stats.pkl"
first_gen = 0
last_gen = 119
save_new_images = True

for gen in tqdm((first_gen, last_gen), desc="Evaluating generations"):
    generation_dir = os.path.join(ip.RESULTS_FOLDER, f"{gen}")
    image_files = [f for f in glob.glob(os.path.join(generation_dir, "*.png")) if "fitness" in os.path.basename(f)]

    population = []
    for img_path in image_files:
        img = Image.open(img_path)
        solution_data = ImageSolutionData(images=[img])
        fitness = evaluator.evaluate(solution_data)
        solution = SolutionCandidate(arguments=None, result=solution_data)
        solution.fitness = fitness
        population.append(solution)
    mo_stats.update_fitness(population)
    if save_new_images: ip.save_images_from_generation(population, gen) # Save images anew with their MO fitness in the filename
    del population
    gc.collect()

    with open(stats_file, "wb") as f:
        pickle.dump(mo_stats, f)

In [ ]:
mo_stats.avg_fitness_stats()

## Radar Charts

In [ ]:
from evolutionary_imaging.processing import create_generation_radar_chart_grid

for gen in (first_gen, last_gen):
    create_generation_radar_chart_grid(gen, tuple(m if isinstance(m, str) else m[0] for m in metrics),
                                       max_images=6,
                                       label_padding=12,
                                       max_value=1.0 # Depends on the objectives set above
                                       ).savefig(f"results/radar_gen_{gen}.pdf")